# VM PA — Can, 데모 prefill + α 수준: `can_prefill_fixa03` × 3 + `can_prefill_t12i_a015` × 3 (vm2_new 판형, 0→9 순서)

전용 G4 VM에서 **0부터 9까지 순서대로**. 1번은 런타임 재시작 → 2번부터. 코드 변경 없음, 6 run(RAM ≈ 102 GB, 5~6시간), 9번이 끝나면 VM 반납. 기준 문서: `HANDOFF.md` §22, 그림 `results/2026-09-12/`.

**앞 VM(9/12)의 결과**: `can_prefill_t12i`는 초기 AUC 0.776 ± 0.007(Can 최고), 129k 0.87, T80 3/3@20k — 그러나 dip은 없어진 게 아니라 **10–15k → 5–10k로 이동**(최저 0.097, mix_prefill 0.093). 5k 바닥은 평가 직전의 상태(α≈0.1, 엔트로피 7–10, |μ|≈0.6)와 9/9 run에서 일치하고(auto-α 과도기가 있든 없든 — 고정 α 0.1도 같음), 데모 90% 배치가 그 비용을 키운다. `can_tent12i_hq`는 해로움(baseline 수준으로 되돌림) → β 스케줄 없음.

## 질문: 데모 배치에서 초기 α/엔트로피 수준을 붙들면 5k와 10–15k의 dip이 **둘 다** 없어지는가
- **주 arm `can_prefill_fixa03_s{1,2,3}`**: `offline_mix.mode=prefill … train.ent_coef=0.3`(고정). fixalpha_03은 과도기 없음(α 0.300 전 행, 5k에 엔트로피 13–16·|μ| 0.33–0.43), Can 후반 엔트로피 13–15 유지(Square형 붕괴 없음), 5k seed 평균 0.468(n=5) / 0.443(s1–3).
- **보조 arm `can_prefill_t12i_a015_s{1,2,3}`**: `… train.ent_coef=auto_0.15 train.target_ent=12` — 과도기 뒤의 α(0.14–0.18)에서 시작해 undershoot만 제거. 주 arm과 같이 보면 "과도 vs 정상상태 α 수준"이 갈린다.

## 사전 판정 (최저 하나로 하지 않는다 — n=3에서 fixalpha_03 자체가 3-seed 부분집합 2/10에서 최저 < 0.405, 그 규칙의 검정력 ≈ 0.6)
- **주 지표 = online 5,008의 seed 평균** ≥ 0.405 (참조: fixalpha_03 s1–3 0.443, prefill_t12i 0.097, mix_prefill 0.78) + seed-matched 차이(vs fixalpha_03 s1–3 기대 ≈ 0, vs prefill_t12i 기대 ≈ +0.35).
- 보조 1: online 10k·15k seed 평균 ≥ 0.405 (mix_prefill형 dip 유무; mix_prefill 0.09/0.30, prefill_t12i 0.37/0.55).
- 보조 2: 초기 AUC — 참조 0.71(= fixalpha_03의 5k점 + mix_prefill 고원) / 0.78(prefill_t12i 고원). 데모는 fixalpha_03 대비 초기 AUC를 안 올렸음(+0.003 ± 0.023)이므로 0.71 근처가 정직한 예상, 0.75 이상이면 prefill_t12i의 고원이 α 0.3에서도 산다는 뜻.
- 보조 3: 129k와 104k/129k 평균 ± SE (예상 ~0.85, SE 0.05–0.07; 데모의 후반 이득은 fixalpha_03 대비 +0.133 ± 0.016로 견고).
- 해석: 주 arm이 5k·10–15k 모두 ≥ 0.405이면 "데모 배치의 dip은 초기 α 수준으로 막힌다"(과도가 아니라 수준). 보조 arm만 통과하면 "과도기가 방아쇠". 둘 다 5k < 0.405이면 데모 배치 고유의 dip → 다음은 critic 타깃 재척도(Q_W +125 → −60) 쪽.
- 비교군(모두 있음): `can_mix_prefill`, `can_prefill_t12i`, `can_fixalpha_03`, `can_tent12i`, `can_calql_prefill`.

평가 격자는 기존과 같은 5k(비교 가능성·비용). dip 폭을 더 보고 싶으면 `eval_schedule.every_env_early=2500`(hardq가 씀)을 붙일 수 있지만 AUC가 격자에 따라 ±0.03 움직이므로 기본은 5k.

## 0. Drive 마운트와 GPU 확인

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Conda 설치 — 실행하면 런타임 자동 재시작

In [ ]:
!pip install -q condacolab
import condacolab
condacolab.install()

## 2. 재시작 후 Drive 재마운트

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
PROJ = '/content/drive/MyDrive/dsrl_project'
print(PROJ)

## 3. 최신 o2o 저장소 준비

In [ ]:
%%bash
set -e
git config --global url."https://github.com/".insteadOf "git@github.com:"
if [ -d /content/dsrl/.git ]; then
  git -C /content/dsrl checkout o2o
  git -C /content/dsrl pull --ff-only origin o2o
else
  test ! -e /content/dsrl || { echo '/content/dsrl exists but is not a git checkout'; exit 2; }
  git clone --recurse-submodules -b o2o https://github.com/msp0617/dsrl.git /content/dsrl
fi
git -C /content/dsrl submodule sync --recursive
git -C /content/dsrl submodule update --init --recursive
echo -n 'HEAD: '; git -C /content/dsrl rev-parse --short HEAD

## 4. 캐시에서 conda 환경 복원

In [ ]:
%%bash
set -e
CACHE=/content/drive/MyDrive/dsrl_project/env_cache/dsrl_env.tar.gz
test -s "$CACHE" || { echo "missing $CACHE"; exit 2; }
mkdir -p /usr/local/envs
rm -rf /usr/local/envs/dsrl
tar -xzf "$CACHE" -C /usr/local/envs
source /usr/local/etc/profile.d/conda.sh
conda activate dsrl
python - <<'PY'
import torch, robomimic, robosuite, mujoco, stable_baselines3
assert torch.cuda.is_available(), 'GPU runtime required'
print('torch', torch.__version__, '| GPU', torch.cuda.get_device_name(0))
PY
python /usr/local/envs/dsrl/lib/python3.10/site-packages/robosuite/scripts/setup_macros.py

## 5. Can 정책·정규화 복원과 환경 패치

In [ ]:
%%bash
set -e
source /usr/local/etc/profile.d/conda.sh && conda activate dsrl
PROJ=/content/drive/MyDrive/dsrl_project
RUNTIME=/content/dsrl/dppo/log
DRIVE=$PROJ/dppo_log
mkdir -p "$RUNTIME"
test -d "$DRIVE" || { echo "missing $DRIVE"; exit 2; }
cp -r "$DRIVE"/. "$RUNTIME"/
CKPT_REL=robomimic-pretrain/can/can_pre_diffusion_mlp_ta4_td20/2024-06-28_13-29-54/checkpoint/state_5000.pt
NORM_REL=robomimic/can/normalization.npz
CKPT_SRC=$(find "$RUNTIME" "$DRIVE" -type f -path "*/$CKPT_REL" -print -quit)
NORM_SRC=$(find "$RUNTIME" "$DRIVE" -type f -path "*/$NORM_REL" -print -quit)
test -n "$CKPT_SRC" -a -n "$NORM_SRC" || { echo 'Can policy assets missing'; exit 2; }
mkdir -p "$RUNTIME/$(dirname "$CKPT_REL")" "$RUNTIME/$(dirname "$NORM_REL")"
[ "$CKPT_SRC" = "$RUNTIME/$CKPT_REL" ] || cp -f "$CKPT_SRC" "$RUNTIME/$CKPT_REL"
[ "$NORM_SRC" = "$RUNTIME/$NORM_REL" ] || cp -f "$NORM_SRC" "$RUNTIME/$NORM_REL"
cat > /content/env.sh <<'EOS'
export MUJOCO_GL=egl
export PYOPENGL_PLATFORM=egl
export WANDB_MODE=disabled
EOS
python /content/dsrl/colab/patch_env.py
ls -lh "$RUNTIME/$CKPT_REL" "$RUNTIME/$NORM_REL"

## 6. 사전검사 — 데모 npz, 이번 exp_id 6개가 비어 있는지(있으면 같은 명령이 checkpoint resume), 비교군, RAM·디스크

In [ ]:
%%bash
set -e
source /usr/local/etc/profile.d/conda.sh && conda activate dsrl
PROJ=/content/drive/MyDrive/dsrl_project
python - <<'PY'
import numpy as np
path = '/content/drive/MyDrive/dsrl_project/offline/can_train_offline.npz'
with np.load(path) as data:
    assert len(data['states']) > 0
    print('OK', path.split('/')[-1], 'rows', len(data['states']))
PY
for E in can_prefill_fixa03_s1 can_prefill_fixa03_s2 can_prefill_fixa03_s3 can_prefill_t12i_a015_s1 can_prefill_t12i_a015_s2 can_prefill_t12i_a015_s3; do
  if [ -f "$PROJ/logs/$E.out" ]; then echo "$E: 이미 있음 -> $(grep '\[done\]\|\[eval\]' $PROJ/logs/$E.out | tail -n 1 | cut -c1-80) (같은 명령이면 resume)"; else echo "$E: 새로 시작"; fi
done
for G in mix_prefill prefill_t12i fixalpha_03 tent12i calql_prefill; do echo -n "비교군 can_$G: "; ls -d $PROJ/logs/can_${G}_s* 2>/dev/null | wc -l; done
echo "processes: $(pgrep -fc '[t]rain_dsrl.py' || true)"; free -g | head -2; df -h /content | tail -n 1

## 7. 6개 시작 — `can_prefill_fixa03` × 3 + `can_prefill_t12i_a015` × 3 (150k, 5k 격자). 주 arm만 돌리려면 두 번째 launch 줄을 지운다

In [ ]:
%%bash
set -e
source /usr/local/etc/profile.d/conda.sh && conda activate dsrl
source /content/env.sh
cd /content/dsrl
git pull --ff-only origin o2o
PROJ=/content/drive/MyDrive/dsrl_project
mkdir -p "$PROJ/logs"
CFG='--config-path=cfg/robomimic --config-name=dsrl_can.yaml'
PREFILL="offline_mix.mode=prefill offline_data_path=$PROJ/offline/can_train_offline.npz"
launch () {
  EXP=$1; shift
  if pgrep -af '[t]rain_dsrl.py' | grep -Fq "exp_id=$EXP"; then echo "already running: $EXP"; return; fi
  nohup python train_dsrl.py $CFG exp_id=$EXP "$@" > "$PROJ/logs/$EXP.out" 2>&1 &
  echo "started $EXP (pid $!)"
}
for S in 1 2 3; do
  launch can_prefill_fixa03_s$S     seed=$S variant=baseline log_dir=$PROJ/logs train.total_env_steps=150000 $PREFILL train.ent_coef=0.3
  launch can_prefill_t12i_a015_s$S  seed=$S variant=baseline log_dir=$PROJ/logs train.total_env_steps=150000 $PREFILL train.ent_coef=auto_0.15 train.target_ent=12
done

## 8. 3분 후 자동 확인 — 6개 running, ERR 없음, 인자에 `train.ent_coef=0.3` / `auto_0.15 … target_ent=12`와 `offline_mix.mode=prefill`. 20~30분 뒤 다시 돌리면 train_log의 α(고정 0.300 / 0.15 근처)·logp·offline_p(≈0.9)도 찍힌다

In [ ]:
%%bash
sleep 180
source /usr/local/etc/profile.d/conda.sh && conda activate dsrl
cd /content/dsrl
PROJ=/content/drive/MyDrive/dsrl_project
python scripts/inspect_runs.py --proj "$PROJ" --only can_prefill_fixa03_s,can_prefill_t12i_a015_s
for E in can_prefill_fixa03_s1 can_prefill_fixa03_s2 can_prefill_fixa03_s3 can_prefill_t12i_a015_s1 can_prefill_t12i_a015_s2 can_prefill_t12i_a015_s3; do
  echo "== $E: $(grep '\[budget\]\|\[eval\]\|Traceback\|Error' "$PROJ/logs/$E.out" | tail -n 2 | tr '\n' ' ' | cut -c1-160)"
  T=$PROJ/logs/$E/train_log.csv
  [ -f "$T" ] && awk -F, 'NR==1{for(i=1;i<=NF;i++)c[$i]=i} END{printf "   train_log last: env_steps=%s ent_coef=%s logp_mean=%s qw_mean=%s mu=%s offline_p=%s\n", $c["env_steps"], $c["ent_coef"], $c["logp_mean"], $c["qw_mean"], $c["mu_absmean"], $c["offline_p"]}' "$T"
done
echo '== processes (인자 확인)'; pgrep -af '[t]rain_dsrl.py' | sed 's/.*exp_id=/exp_id=/' | cut -c1-200 || true
free -g | head -2

## 9. Keepalive — 마지막 프로세스가 끝나면 자동 반납

8번에서 오류가 없을 때만 실행하고 이 셀을 계속 실행 상태로 둡니다.

In [ ]:
import subprocess, time
from pathlib import Path
EXPECTED = [f'can_{kind}_s{s}' for kind in ('prefill_fixa03', 'prefill_t12i_a015') for s in (1, 2, 3)]
LOGS = Path('/content/drive/MyDrive/dsrl_project/logs')

def running():
    out = subprocess.run(['ps', '-eo', 'pid,args'], capture_output=True, text=True).stdout
    return [line.strip() for line in out.splitlines() if 'train_dsrl.py' in line and any(f'exp_id={e}' in line for e in EXPECTED)]

def last_event(exp):
    path = LOGS / f'{exp}.out'
    if not path.exists(): return 'NO .out'
    lines = path.read_text(errors='replace').splitlines()[-500:]
    for line in reversed(lines):
        if any(x in line for x in ('[eval]', '[done]', 'Traceback', 'Error')): return line[:100]
    return 'starting'

while True:
    procs = running()
    print(time.strftime('%H:%M'), f'running {len(procs)}/{len(EXPECTED)}', '|', ' | '.join(f'{e}: {last_event(e)}' for e in EXPECTED), flush=True)
    if not procs:
        print('all VM PA runs stopped -> unassigning', flush=True)
        from google.colab import runtime
        runtime.unassign()
        break
    time.sleep(600)

## 10. 결과 zip (끝난 뒤, CPU 런타임 + 0번 Drive 마운트만으로 됨). 로컬에서 `~/Downloads/logs/`에 풀고
`.venv/bin/python scripts/plot_results.py --logs ~/Downloads/logs --out ~/Downloads/dsrl_figs_pa --axes "prefill_alpha=baseline,mix_prefill,prefill_t12i,prefill_fixa03,prefill_t12i_a015;alpha_ref=fixalpha_03,tent12i,prefill_fixa03,prefill_t12i_a015"`

In [ ]:
%%bash
cd /content/drive/MyDrive/dsrl_project
rm -f csv_bundle.zip
zip -qr csv_bundle.zip logs -i "logs/*/eval_log.csv" "logs/*/train_log.csv" "logs/*.csv" "logs/pretrain/*_log.csv"
ls -lh csv_bundle.zip